# Train the GeM compliance models

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough).
2. **Runtime → Run all.** Full run: roughly an hour on a T4.
3. Set `HF_USER` in the last section to publish the models to Hugging Face.

Everything below is also documented in `ml/README.md` of the repository.

In [ ]:
REPO = "https://github.com/harshil-777/SIH_26100v1.git"
BRANCH = "ml-model"
# Private repository? Use a GitHub personal access token (read-only is enough):
# REPO = "https://<YOUR_GITHUB_TOKEN>@github.com/harshil-777/SIH_26100v1.git"

!git clone -q -b $BRANCH $REPO repo
%cd repo
!pip install -q -r ml/requirements-ml.txt

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to a GPU")

## 1. Certificate field extraction (DistilBERT token classification)

In [ ]:
!python -m ml.extraction.generate --n 60000
!python -m ml.extraction.train --epochs 3
!python -m ml.extraction.evaluate

## 2. Bid risk estimator (LightGBM — CPU is fine)

In [ ]:
!python -m ml.risk.generate --n 200000
!python -m ml.risk.train

## 3. Recommendation writer (flan-t5-small)

In [ ]:
!python -m ml.recommendation.generate --n 60000
!python -m ml.recommendation.train --epochs 4

## 4. Publish to Hugging Face

Needs a **write** token from https://huggingface.co/settings/tokens. Add `--private` to keep a repo private.

In [ ]:
HF_USER = ""  # your Hugging Face username

from huggingface_hub import notebook_login
notebook_login()

In [ ]:
assert HF_USER, "Set HF_USER above first"
!python -m ml.push_to_hub --model extraction --repo {HF_USER}/gem-certificate-extractor
!python -m ml.push_to_hub --model risk --repo {HF_USER}/gem-bid-risk --with-dataset
!python -m ml.push_to_hub --model recommendation --repo {HF_USER}/gem-recommendation-writer

## Optional: download the trained models as a zip

In [ ]:
!cd ml && zip -qr ../trained_models.zip models -x "*/checkpoints/*"
from google.colab import files
files.download("trained_models.zip")